In [3]:
!pip install -q transformers datasets sentencepiece accelerate evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 14.2 MB/s eta 0:00:00


In [4]:
import torch
import transformers
import datasets

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("Datasets version:", datasets.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.10.0+cu128
Transformers version: 5.0.0
Datasets version: 4.0.0
GPU available: True


In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
GPU available: True
GPU: Tesla T4


In [6]:
from datasets import load_dataset

dataset = load_dataset("cfilt/iitb-english-hindi")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})


In [7]:
print(dataset["train"][0])

{'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}}


In [8]:
print("English:", dataset["train"][0]["translation"]["en"])
print("Hindi:", dataset["train"][0]["translation"]["hi"])

English: Give your application an accessibility workout
Hindi: अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें


In [9]:
print("Training data:", len(dataset["train"]))
print("Validation data:", len(dataset["validation"]))
print("Test data:", len(dataset["test"]))

Training data: 1659083
Validation data: 520
Test data: 2507


In [10]:
train_data = dataset["train"].select(range(10000))

print("Training examples:", len(train_data))

Training examples: 10000


In [11]:
val_data = dataset["validation"].select(range(520))
test_data = dataset["test"].select(range(1000))

print("Validation examples:", len(val_data))
print("Test examples:", len(test_data))

Validation examples: 520
Test examples: 1000


In [12]:
print(train_data[0])
print(val_data[0])
print(test_data[0])

{'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}}
{'translation': {'en': 'Students of the Dattatreya city Municipal corporation secondary school demonstrated their imagination power by creating the fictitious fort "Duttgarh".', 'hi': "महानगर पालिका अंतर्गत दत्तात्रय नगर माध्यमिक स्कूल के विद्यार्थियों ने काल्पनिक किला 'दत्तगढ़' बनाकर अपनी कल्पनाशक्ति का परिचय दिया।"}}
{'translation': {'en': 'A black box in your car?', 'hi': 'आपकी कार में ब्लैक बॉक्स?'}}


In [13]:
from transformers import AutoTokenizer

model_name = "Helsinki-NLP/opus-mt-en-hi"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully!


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [14]:
text = "How are you?"

tokens = tokenizer(text)

print(tokens)

{'input_ids': [244, 54, 27, 22, 0], 'attention_mask': [1, 1, 1, 1, 1]}


In [15]:
hindi_text = "आप कैसे हैं?"

tokens = tokenizer(hindi_text)

print(tokens)

{'input_ids': [44, 3605, 2703, 44, 22216, 44, 6286, 549, 22, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [16]:
def preprocess_function(examples):
    inputs = [item["en"] for item in examples["translation"]]
    targets = [item["hi"] for item in examples["translation"]]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [17]:
tokenized_train = train_data.map(
    preprocess_function,
    batched=True
)

print(tokenized_train)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset({
    features: ['translation', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 10000
})


In [18]:
tokenized_val = val_data.map(
    preprocess_function,
    batched=True
)

tokenized_test = test_data.map(
    preprocess_function,
    batched=True
)

print("Training:", len(tokenized_train))
print("Validation:", len(tokenized_val))
print("Test:", len(tokenized_test))

Map:   0%|          | 0/520 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Training: 10000
Validation: 520
Test: 1000


In [19]:
print(tokenized_train[0])

{'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}, 'input_ids': [3872, 85, 2501, 132, 15441, 36398, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1], 'labels': [63, 2025, 18, 16155, 346, 20311, 24, 2279, 679, 0]}


In [20]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Transformer model loaded successfully!")

pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Transformer model loaded successfully!


In [21]:
print(type(model))

<class 'transformers.models.marian.modeling_marian.MarianMTModel'>


In [22]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

print("Data collator created successfully!")

Data collator created successfully!


In [23]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [24]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./english_hindi_model",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    report_to="none"
)

print("Training configuration created!")

Training configuration created!


In [25]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

print("Trainer created successfully!")

Trainer created successfully!


In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.550503,4.429170


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.8046940780639649, metrics={'train_runtime': 126.0667, 'train_samples_per_second': 79.323, 'train_steps_per_second': 9.915, 'total_flos': 29843273023488.0, 'train_loss': 0.8046940780639649, 'epoch': 1.0})

In [27]:
results = trainer.evaluate()

print(results)

{'eval_loss': 4.42917013168335, 'eval_runtime': 1.708, 'eval_samples_per_second': 304.458, 'eval_steps_per_second': 38.057, 'epoch': 1.0}


In [28]:
text = "The patient has a fever."

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_length=128
)

translation = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("English:", text)
print("Hindi:", translation)

English: The patient has a fever.
Hindi: मरीज़ को बुखार है.


In [29]:
sentences = [
    "The patient has a fever.",
    "The doctor examined the patient.",
    "The computer is connected to the network.",
    "The sample was collected from the laboratory."
]

for text in sentences:
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=128)

    translation = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("English:", text)
    print("Hindi:", translation)
    print("-" * 50)

English: The patient has a fever.
Hindi: मरीज़ को बुखार है.
--------------------------------------------------
English: The doctor examined the patient.
Hindi: डॉक्टर ने मरीज़ की जाँच की ।
--------------------------------------------------
English: The computer is connected to the network.
Hindi: कम्प्यूटर नेटवर्क से जुड़ा हुआ है.
--------------------------------------------------
English: The sample was collected from the laboratory.
Hindi: नमूना प्रयोगशाला से इकट्ठा किया गया था.
--------------------------------------------------


In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.398839,4.502147


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.43009022216796877, metrics={'train_runtime': 120.815, 'train_samples_per_second': 82.771, 'train_steps_per_second': 10.346, 'total_flos': 29843273023488.0, 'train_loss': 0.43009022216796877, 'epoch': 1.0})

In [31]:
print(model_name)
print(type(tokenizer))
print(type(model))
print(len(tokenized_train))
print(len(tokenized_val))

Helsinki-NLP/opus-mt-en-hi
<class 'transformers.models.marian.tokenization_marian.MarianTokenizer'>
<class 'transformers.models.marian.modeling_marian.MarianMTModel'>
10000
520


In [32]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-hi"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Tokenizer and model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Tokenizer and model loaded successfully!


In [33]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

print("Data collator ready!")

Data collator ready!


In [34]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./english_hindi_model",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    report_to="none"
)

print("Training arguments ready!")

Training arguments ready!


In [35]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

print("Trainer created successfully!")

Trainer created successfully!


In [36]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.550503,4.429170


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.8046940719604492, metrics={'train_runtime': 128.5453, 'train_samples_per_second': 77.794, 'train_steps_per_second': 9.724, 'total_flos': 29843273023488.0, 'train_loss': 0.8046940719604492, 'epoch': 1.0})

In [37]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.398839,4.502148


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.4300902084350586, metrics={'train_runtime': 128.2921, 'train_samples_per_second': 77.947, 'train_steps_per_second': 9.743, 'total_flos': 29843273023488.0, 'train_loss': 0.4300902084350586, 'epoch': 1.0})

In [38]:
trainer.save_model("./english_hindi_model")
tokenizer.save_pretrained("./english_hindi_model")

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [39]:
results = trainer.evaluate()

print(results)

{'eval_loss': 4.502147674560547, 'eval_runtime': 3.2191, 'eval_samples_per_second': 161.535, 'eval_steps_per_second': 20.192, 'epoch': 1.0}


In [40]:
import evaluate

bleu = evaluate.load("sacrebleu")

print("BLEU metric loaded successfully!")

BLEU metric loaded successfully!


In [41]:
def translate_text(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=128
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [42]:
text = "The patient has a fever."

translation = translate_text(text)

print("English:", text)
print("Hindi:", translation)

English: The patient has a fever.
Hindi: मरीज़ को बुखार है.


In [43]:
sentences = [
    "The patient has a fever.",
    "The doctor examined the patient.",
    "The sample was collected from the laboratory.",
    "The computer is connected to the network."
]

for sentence in sentences:
    print("English:", sentence)
    print("Hindi:", translate_text(sentence))
    print("-" * 60)

English: The patient has a fever.
Hindi: मरीज़ को बुखार है.
------------------------------------------------------------
English: The doctor examined the patient.
Hindi: डॉक्टर ने मरीज़ की जाँच की ।
------------------------------------------------------------
English: The sample was collected from the laboratory.
Hindi: नमूना प्रयोगशाला से इकट्ठा किया गया था.
------------------------------------------------------------
English: The computer is connected to the network.
Hindi: कम्प्यूटर नेटवर्क से जुड़ा हुआ है.
------------------------------------------------------------


In [44]:
import evaluate

bleu = evaluate.load("sacrebleu")

predictions = []
references = []

for i in range(100):
    english_text = test_data[i]["translation"]["en"]
    hindi_text = test_data[i]["translation"]["hi"]

    prediction = translate_text(english_text)

    predictions.append(prediction)
    references.append([hindi_text])

print("Translations generated:", len(predictions))

Translations generated: 100


In [45]:
bleu_result = bleu.compute(
    predictions=predictions,
    references=references
)

print("BLEU Score:", bleu_result["score"])

BLEU Score: 7.65652739335365


In [46]:
bleu_score = bleu_result["score"]

print(f"Final BLEU Score: {bleu_score:.2f}")

Final BLEU Score: 7.66


In [47]:
import pandas as pd

results_df = pd.DataFrame({
    "English": [
        test_data[i]["translation"]["en"] for i in range(5)
    ],
    "Actual Hindi": [
        test_data[i]["translation"]["hi"] for i in range(5)
    ],
    "Predicted Hindi": predictions[:5]
})

results_df

,English,Actual Hindi,Predicted Hindi
0,A black box in your car?,आपकी कार में ब्लैक बॉक्स?,अपनी कार में एक ब्लैक बॉक्स?
1,As America's road planners struggle to find th...,"जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए...",के रूप में अमेरिका के सड़क योजनार एक बंद सड़क ...
2,"The devices, which track every mile a motorist...","यह डिवाइस, जो मोटर-चालक द्वारा वाहन चलाए गए प्...","ये उपकरण हैं, जो हर मील एक मोटर ड्राइव और भेज ..."
3,The usually dull arena of highway planning has...,आम तौर पर हाईवे नियोजन जैसा उबाऊ काम भी अचानक ...,आम तौर पर सड़क योजना का नया तरीका शुरू हो गया ...
4,Libertarians have joined environmental groups ...,"आपने द्वारा ड्राइव किए गए मील, तथा संभवतः ड्रा...","और शायद जहाँ आप जाते हैं, वहाँ कर के बिल भरने ..."


In [48]:
import evaluate

bleu = evaluate.load("sacrebleu")

predictions = []
references = []

for i in range(100):
    english_text = test_data[i]["translation"]["en"]
    hindi_text = test_data[i]["translation"]["hi"]

    prediction = translate_text(english_text)

    predictions.append(prediction)
    references.append([hindi_text])

print("Translations generated:", len(predictions))

Translations generated: 100


In [49]:
bleu_result = bleu.compute(
    predictions=predictions,
    references=references
)

print("BLEU Score:", bleu_result["score"])

BLEU Score: 7.65652739335365


In [50]:
bleu_score = bleu_result["score"]

print(f"Final BLEU Score: {bleu_score:.2f}")

Final BLEU Score: 7.66


In [51]:
model_path = "./english_hindi_model"

model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print("Model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!


In [52]:
def translate(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=128
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [53]:
text = input("Enter an English sentence: ")

result = translate(text)

print("English:", text)
print("Hindi:", result)

Enter an English sentence: The doctor examined the patient.
English: The doctor examined the patient.
Hindi: डॉक्टर ने मरीज़ की जाँच की ।


In [54]:
sample_sentences = [
    "The patient has a fever.",
    "The doctor examined the patient.",
    "The sample was collected from the laboratory.",
    "The computer is connected to the network."
]

result_data = []

for sentence in sample_sentences:
    hindi = translate(sentence)

    result_data.append({
        "English": sentence,
        "Predicted Hindi": hindi
    })

import pandas as pd

results_table = pd.DataFrame(result_data)
results_table

,English,Predicted Hindi
0,The patient has a fever.,मरीज़ को बुखार है.
1,The doctor examined the patient.,डॉक्टर ने मरीज़ की जाँच की ।
2,The sample was collected from the laboratory.,नमूना प्रयोगशाला से इकट्ठा किया गया था.
3,The computer is connected to the network.,कम्प्यूटर नेटवर्क से जुड़ा हुआ है.


In [55]:
!pip install -q gradio

In [56]:
import gradio as gr

def translate_for_ui(text):
    if not text.strip():
        return ""

    return translate(text)

demo = gr.Interface(
    fn=translate_for_ui,
    inputs=gr.Textbox(
        label="English Text",
        placeholder="Enter an English sentence..."
    ),
    outputs=gr.Textbox(
        label="Hindi Translation"
    ),
    title="English → Hindi Machine Translation",
    description="Transformer-based English to Hindi translation system"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3a2a6376169d0f56a9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [57]:
 print(f"BLEU Score: {bleu_score:.2f}")

BLEU Score: 7.66
